# Apply the run-all fixes (v2)

Use this instead of `00_apply_fixes.ipynb`, which located cells by absolute index
and therefore broke once a cell was skipped or removed in 06 — and worse, aborted on
that first miss, leaving 07–09 unpatched.

This version finds each target by its own text, so cell numbering no longer matters.
It is safe to re-run: already-patched cells are reported and left alone. A target
that no longer exists is skipped rather than treated as an error — the text it looks
for *is* the failure site, so its absence means there is nothing left to protect.

It patches the notebooks already in `MyDrive/algoverse/dnabert/` **in place**. It does
not re-stage from the zips, so completed work and everything in `outputs/` is
untouched.

**Close any of 06–09 you have open first**, then reopen after this finishes —
otherwise Colab's in-memory copy overwrites the patch on its next autosave.

| notebook | what raised | why passing it is safe |
|---|---|---|
| 06 4A | smoke test pools a 3-D tensor; DNABERT-2 returns flattened 2-D | 4B redefines the pooler and reruns it; the real scan uses the fixed one |
| 07 c6 | asserts `(source_group, k)` unique, but each source appears at two mutation levels | the next cell re-validates on the correct key and saves |
| 08 c8 | prints `.coef_.shape`; the probe is a pipeline, so `coef_` is absent | diagnostic print only, made tolerant |
| 08 c9 | prints `outputs.hidden_states`; the model returns a plain tuple | the next cell inspects it properly |
| 09 c6 | raises on any unexpected checkpoint key; the checkpoint carries unused MLM-head weights | the next cell's allow-list (`cls.predictions.`) applied here |

Note on 06: the smoke-test call appears in **both** 4A and 4B, since 4B reruns it with
the corrected pooler. Only 4A is ever wrapped; 4B is explicitly excluded.

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
ROOT = "/content/drive/MyDrive/algoverse/dnabert"

exec(open(f"{ROOT}/_patches_v2.py").read())

print("Patching in place:")
applied, skipped = apply_patches(ROOT)

residual = verify_patches(ROOT)
print(f"residual fatal patterns: {residual}")
assert residual == 0, "a known-fatal pattern is still reachable"
print("\nDone. Reopen 07-09 to pick up the patched copies.")

Patching in place:
  06_gc_controlled_interventions.ipynb                 target absent (cell edited or removed) - skipped
  07_low_rank_subspace_analysis.ipynb                  already patched
  08_promoter_mlp_subspace_causal_interventions.ipynb  already patched
  08_promoter_mlp_subspace_causal_interventions.ipynb  already patched
  09_promoter_pathway_mediation_validation.ipynb       already patched

applied 0, skipped 5
residual fatal patterns: 0

Done. Reopen 07-09 to pick up the patched copies.
